# EEG and ECG preprocessing

Select **Kernel → Change Kernel → Python (dbh)**, then run the cells in order.
If the notebook was already open during setup, reload it before selecting the kernel.
This notebook calls the
existing Python and MATLAB functions for the selected subjects. The real `sub-001`
example can be processed without `participants.csv`.

**First-time setup**, from the repository root in a terminal:

```bash
conda env create -f environment.yml
conda activate dbh
python -m ipykernel install --user --name dbh --display-name "Python (dbh)"
python -m jupyterlab 01_preprocessing/run_preprocessing.ipynb
```

For the EEG workflow, install MATLAB with Signal Processing Toolbox, EEGLAB with BVA-io, DIPFIT,
Zapline-plus, CleanLine, ICLabel and ERPLAB, and FieldTrip. Put the toolboxes in
`toolboxes/eeglab` and `toolboxes/fieldtrip`, or enter their installed locations below.
See the [setup instructions](../README.md#setup). MATLAB steps launch the installed
MATLAB in batch mode, display its console output here, and wait for completion.

Run **shared setup (steps 1–2)**, then either workflow or both:

- **EEG (steps 3–4):** check MATLAB → clean EEG → compute band power.
- **ECG (steps 5–7):** extract ECG → reuse or review R peaks → compute IBI/HRV.

The ECG workflow does not require MATLAB or EEG outputs.
When a step finds existing results for the selected subjects, it asks:
**1 = skip existing results (default); 2 = recompute and replace them.**
Missing outputs are still created. The choice applies only to that step and the
selected subjects. For R peaks, option 2 reopens the editor; files are replaced
only when you save. There is no separate review-confirmation prompt.


## 1. Choose single-subject or batch processing

Run the next cell and answer the prompts below it. **Press Enter to accept each
displayed default, or type a replacement and press Enter.** A visible input box
means the notebook is waiting for your answer.

**Included batch example:** enter **2**, press **Enter** for the participant CSV,
then press **Enter** for the raw recording folder.

- **1 — single subject:** enter one ID; press Enter for the `sub-001` example.
- **2 — batch from CSV:** choose a participant information table. Its `ID` column
  supplies the IDs to look for, in table order. Blank or duplicate IDs are rejected.
- Confirm the raw recording folder. Relative paths are resolved from the repository
  root. Step 2 looks for `{ID}_movie.vhdr`, `.eeg` and `.vmrk` for each listed ID.

The included default table, `00_data/participants.csv`, lists all 148 recoded study IDs.
Only the `sub-001` recording is included; a full batch requires the remaining recordings.
Table IDs must match the recording filenames. Step 2 prints `Missing` for IDs
with any missing recording file and skips them. All subsequent steps use only
the available subjects, in CSV order. If none are available, processing is skipped.

`MATLAB_EXE = None` finds MATLAB automatically. Set the toolbox locations below
when needed. The selected recording folder is passed to both Python and MATLAB.
Changing mode requires rerunning this cell and the subsequent steps.

Outputs go to `01_preprocessing/outputs/demo/` for one subject or
`01_preprocessing/outputs/batch/` for a CSV batch. The supplied manuscript
arrays in `01_preprocessing/outputs/manuscript/` are read only and are never
replaced by this notebook.


In [3]:
from importlib import import_module, reload
from IPython.display import HTML, display
from pathlib import Path
import os
import re
import shutil
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "01_preprocessing/config/preprocessing_config.m").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository.")
PRE = ROOT / "01_preprocessing"
CONFIG_DIR = PRE / "config"
EEG_DIR = PRE / "eeg"
ECG_DIR = PRE / "ecg"
OUTPUTS = None  # Step 1 selects outputs/demo (one subject) or outputs/batch (CSV batch).
display(HTML('<style id="dbh-notebook-style">' +
             (ROOT / "common/notebook_style.css").read_text(encoding="utf-8") + "</style>"))
SUBJECTS = None
# Toolbox locations: toolboxes/ at the repository root (real folders or symbolic links),
# or the environment variables DBH_EEGLAB_PATH / DBH_FIELDTRIP_PATH set before Jupyter starts.
EEGLAB_DIR = os.environ.get("DBH_EEGLAB_PATH") or ROOT / "toolboxes/eeglab"
FIELDTRIP_DIR = os.environ.get("DBH_FIELDTRIP_PATH") or ROOT / "toolboxes/fieldtrip"
MATLAB_EXE = None

EEGLAB_DIR = Path(EEGLAB_DIR).expanduser()
FIELDTRIP_DIR = Path(FIELDTRIP_DIR).expanduser()
if not EEGLAB_DIR.is_absolute():
    EEGLAB_DIR = ROOT / EEGLAB_DIR
if not FIELDTRIP_DIR.is_absolute():
    FIELDTRIP_DIR = ROOT / FIELDTRIP_DIR

for folder in (ROOT, CONFIG_DIR):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
import select_subjects
reload(select_subjects)
RunLog = reload(import_module("common.run_log")).RunLog
LOG = RunLog(PRE / "logs", ROOT)

with LOG.step("Subject selection"):
    MODE, REQUESTED_SUBJECTS, RAW_DATA_DIR = select_subjects.select_subjects(ROOT)
    SUBJECTS = []  # Step 2 keeps only subjects with complete recording files.
    OUTPUTS = PRE / "outputs" / ("batch" if MODE == "batch" else "demo")
    HRV_OUTPUT = OUTPUTS / "ecg/hrv" / ("ecg_ts.npz" if MODE == "batch" else f"ecg_ts_{REQUESTED_SUBJECTS[0]}.npz")
    sys.path.insert(0, str(ECG_DIR))
    sys.path.insert(0, str(ECG_DIR / "ECG_R_peak_editor"))
    print("Repository:", ROOT)
    print("Requested subjects:", ", ".join(REQUESTED_SUBJECTS))
    print("Output folder:", LOG.display(OUTPUTS))
    print("HRV output:", LOG.display(HRV_OUTPUT))
    print("Python:", sys.executable)
    LOG.write(f"Mode: {MODE}; requested subjects: {len(REQUESTED_SUBJECTS)}")
    LOG.write(f"Raw recordings: {RAW_DATA_DIR}")
    LOG.write(f"Output folder: {OUTPUTS}")


Run log: 01_preprocessing/logs/20261030_231526.log
Choose a processing mode:
1 = single subject
2 = batch from participant CSV


Type 1 or 2, then press Enter (Enter alone = 1):  1
Subject ID (press Enter to use sub-001):  sub-001



Raw recording folder
Default folder: 00_data/raw_data


Press Enter to use this folder, or type another folder path:  


Mode: single; selected subjects: 1
Raw recording folder: 00_data/raw_data
Repository: /Users/chen/Library/CloudStorage/OneDrive-Personal/Chen_/academic_project/Xie/Dev_Brain_Heart/DBH_github_files
Requested subjects: sub-001
Output folder: 01_preprocessing/outputs/demo
HRV output: 01_preprocessing/outputs/demo/ecg/hrv/ecg_ts_sub-001.npz
Python: /opt/anaconda3/envs/dbh/bin/python


Run this helper cell once. It resolves paths, appends to the run log, and asks
whether to skip or replace existing results. The code is collapsed; choices appear
in the output area of each processing step below.


In [5]:
from output_policy import choose_overwrite as _choose_overwrite, has_output


def choose_overwrite(outputs, label, **kwargs):
    overwrite = _choose_overwrite(outputs, label, **kwargs)
    LOG.write("Existing outputs: replace" if overwrite else "Existing outputs: keep; create missing")
    return overwrite


def available_subjects(ids, raw_dir):
    """Keep complete BrainVision recordings in the requested order."""
    raw_dir = Path(raw_dir)
    available = []
    for sid in ids:
        missing = [f".{ext}" for ext in ("vhdr", "eeg", "vmrk")
                   if not (raw_dir / f"{sid}_movie.{ext}").is_file()]
        if missing:
            print(f"Missing: {sid} ({', '.join(missing)}); skipped")
        else:
            available.append(sid)
    print(f"Ready: {len(available)}; Missing: {len(ids) - len(available)}")
    missing_ids = [sid for sid in ids if sid not in available]
    LOG.write(f"Available ({len(available)}): {', '.join(available) or 'none'}")
    if missing_ids:
        LOG.write(f"Missing ({len(missing_ids)}), skipped: {', '.join(missing_ids)}")
    return available


def require_files(paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        preview = "\n".join(missing[:10])
        raise FileNotFoundError(f"Missing {len(missing)} file(s); first 10 shown:\n{preview}")


def needs_run(output, inputs, overwrite=False):
    require_files(inputs)
    output = Path(output)
    if overwrite or not has_output(output):
        return True
    if output.suffix == ".mat":
        import h5py
        if not h5py.is_hdf5(output):
            raise ValueError(f"{output.name} is not a valid result. Rerun this step and choose 2 to replace it.")
    print("Skip existing:", output.name)
    LOG.write(f"Kept: {output}")
    return False


def find_matlab():
    if MATLAB_EXE:
        return str(MATLAB_EXE)
    found = shutil.which("matlab")
    if found:
        return found
    patterns = []
    if sys.platform == "darwin":
        patterns = list(Path("/Applications").glob("MATLAB_R*.app/bin/matlab"))
    elif os.name == "nt":
        base = Path(os.environ.get("ProgramFiles", "C:/Program Files"))
        patterns = list((base / "MATLAB").glob("R*/bin/matlab.exe"))
    if patterns:
        return str(sorted(patterns)[-1])
    raise FileNotFoundError("MATLAB was not found. Set MATLAB_EXE in step 1.")


def matlab_quote(value):
    return "'" + str(value).replace("'", "''") + "'"


def run_matlab(command):
    command = re.sub(r"\.\.\.\s*\n\s*", " ", command)
    command = "; ".join(line.strip().rstrip(";") for line in command.splitlines() if line.strip())
    env = os.environ.copy()
    env["DBH_RAW_DATA_PATH"] = str(RAW_DATA_DIR)
    env["DBH_PREPROCESSING_OUTPUT"] = str(OUTPUTS)
    env["DBH_EEGLAB_PATH"] = str(Path(EEGLAB_DIR).resolve())
    env["DBH_FIELDTRIP_PATH"] = str(Path(FIELDTRIP_DIR).resolve())
    prefix = (
        f"addpath({matlab_quote(CONFIG_DIR)}); addpath({matlab_quote(EEG_DIR)}); "
        "CFG=preprocessing_config(); addpath(CFG.eeglabPath); "
        "addpath(CFG.fieldtripPath); "
    )
    args = [find_matlab()]
    if os.name == "nt":
        args.append("-wait")
    args += ["-sd", str(ROOT), "-batch", prefix + command + "; disp('DBH_STEP_COMPLETE');"]
    LOG.command(args, env=env, completion_marker="DBH_STEP_COMPLETE")


def run_matlab_step(function, output_for, inputs_for):
    if not SUBJECTS:
        print("No available subjects; skipped.")
        LOG.write("Skipped: no available subjects")
        return
    source = EEG_DIR / f"{function}.m"
    sources = [source, CONFIG_DIR / "preprocessing_config.m"]
    overwrite = choose_overwrite([output_for(sid) for sid in SUBJECTS], function)
    todo = [sid for sid in SUBJECTS
            if needs_run(output_for(sid), inputs_for(sid) + sources, overwrite)]
    if todo:
        LOG.write("Processing: " + ", ".join(todo))
        ids = ",".join(matlab_quote(sid) for sid in todo)
        command = f"ids={{{ids}}}; for k=1:numel(ids), {function}(ids{{k}}); end"
        run_matlab(command)
    else:
        print(function, ": existing results kept")
        LOG.write("Skipped processing: all selected outputs already exist")
    require_files([output_for(sid) for sid in SUBJECTS])

## 2. Check Python and the input files

Missing recordings are listed and skipped. Only subjects with all three
BrainVision files remain in `SUBJECTS`. One log is saved under
`01_preprocessing/logs/YYYYMMDD_HHMMSS.log`. Rerunning step 1 starts a new log;
rerunning later cells appends to the same file. It records selected subjects,
processing parameters, concise EEG/ECG summaries, skipped outputs, warnings and
errors. Full external-program output remains visible in the notebook.

If NumPy/Numba imports fail, check the Python path printed in step 1. Select
**Python (dbh)** and restart from step 1. MATLAB is checked separately in the EEG
workflow; after this cell you can run EEG, ECG, or both.


In [7]:
with LOG.step("Inputs and Python setup"):
    SUBJECTS = available_subjects(REQUESTED_SUBJECTS, RAW_DATA_DIR)
    if SUBJECTS:
        LOG.command([sys.executable, ROOT / "check_setup.py", "--python-only"])
    else:
        print("No available subjects; processing is skipped.")
        LOG.write("Skipped: no available subjects")


Ready: 1; Missing: 0
Python 3.12.7: /opt/anaconda3/envs/dbh/bin/python
[OK] numpy: 2.2.6
[OK] scipy: 1.16.3
[OK] pandas: 2.3.3
[OK] matplotlib: 3.10.8
[OK] sklearn: 1.8.0
[OK] statsmodels: 0.14.6
[OK] h5py: 3.16.0
[OK] joblib: 1.4.2
[OK] numba: 0.61.2
[OK] llvmlite: 0.44.0
[OK] tqdm: 4.66.5
[OK] mne: 1.11.0
[OK] neurokit2: 0.2.13
[OK] IPython: 9.17.1
[OK] ipykernel: 6.28.0
[OK] jupyterlab: 4.2.5
[OK] nbformat: 5.10.4
[OK] tkinter: available
MATLAB, its plugins, recording contents and downstream analyses were not tested.
Requested setup checks passed. Follow the analysis order in the READMEs.


## EEG workflow

### 3. Check MATLAB and clean EEG

The first cell checks the MATLAB license and toolbox functions. It does not process
recordings. Skip this entire EEG section if you only need ECG/HRV.


In [9]:
with LOG.step("MATLAB setup"):
    if SUBJECTS:
        LOG.write(f"MATLAB: {find_matlab()}")
        LOG.write(f"EEGLAB: {EEGLAB_DIR}; FieldTrip: {FIELDTRIP_DIR}")
        require_files([Path(EEGLAB_DIR) / "eeglab.m", Path(FIELDTRIP_DIR) / "ft_defaults.m"])
        run_matlab("""
        assert(license('test','Signal_Toolbox'), 'Signal Processing Toolbox license is required');
        addpath(genpath(CFG.eeglabPath)); eeglab nogui;
        addpath(CFG.fieldtripPath); ft_defaults;
        required={'pop_loadbv','pop_zapline_plus','pop_cleanline','pop_basicfilter', ...
        'pop_runica','iclabel','pop_icflag','pop_subcomp','eeg_interp','pop_reref', ...
        'eeglab2fieldtrip','pop_creabasiceventlist','pop_binlister','pop_epochbin', ...
        'pop_artextval','pop_artmwppth','ft_freqanalysis'};
        for k=1:numel(required)
            assert(exist(required{k},'file')>0, 'Missing toolbox function: %s', required{k});
        end;
        assert(~isempty(ft_senslabel('ant128')), 'FieldTrip sensor helpers did not load');
        fprintf('[summary] MATLAB %s: preprocessing dependencies found.\\n', version)
        """)
    else:
        print("No available subjects; skipped.")
        LOG.write("Skipped: no available subjects")


you rename the function to avoid a potential name conflict. 
you rename the function to avoid a potential name conflict. 

Path Warning: It appears that you have added the path to all of the
subfolders to EEGLAB. This may create issues with some EEGLAB extensions
If EEGLAB cannot start or your experience a large number of warning
messages, remove all the EEGLAB paths then go to the EEGLAB folder
and start EEGLAB which will add all the necessary paths.

 
eeglab: options file is ~/eeg_options.m
Retrieving plugin versions from server...
Retrieving download statistics...
EEGLAB: adding "EEG-BIDS" v? (see >> help eegplugin_eegbids) - new version 10.5 available
EEGLAB: adding "ICLabel" v? (see >> help eegplugin_iclabel) - new version 1.7 available (critical bug fix)
EEGLAB: adding "bva-io" v1.74 (see >> help eegplugin_bva_io) - new version 1.75 available
EEGLAB: adding "clean_rawdata" v? (see >> help eegplugin_clean_rawdata) - new version 2.11 available
EEGLAB: adding "cleanline-master" v? 

The next cell removes the two non-EEG channels, applies line-noise removal and
filtering, selects movie segments, fits ICA, rejects labelled artifacts,
interpolates removed channels, and applies average reference. ICA is the longest
step; progress appears below.
Output: `<output folder>/eeg/cleaned/{ID}_continuous_clean.mat`.


In [ ]:
with LOG.step("EEG cleaning"):
    run_matlab_step(
        "preprocess_eeg",
        lambda sid: OUTPUTS / "eeg/cleaned" / f"{sid}_continuous_clean.mat",
        lambda sid: [RAW_DATA_DIR / f"{sid}_movie.{ext}" for ext in ("vhdr", "eeg", "vmrk")],
    )



preprocess_eeg: 1 existing result file(s)
  sub-001_continuous_clean.mat
1 = skip existing results; create only missing results (default)
2 = recompute and replace


### 4. Compute EEG band power

This applies the existing artifact mask and STFT procedure for delta, theta, alpha
and beta power. Output: `<output folder>/eeg/band_power/{ID}_movie_EEG_TS.mat`.

## with LOG.step("EEG band power"):
    run_matlab_step(
        "compute_eeg_band_power",
        lambda sid: OUTPUTS / "eeg/band_power" / f"{sid}_movie_EEG_TS.mat",
        lambda sid: [OUTPUTS / "eeg/cleaned" / f"{sid}_continuous_clean.mat"],
    )


## ECG workflow

### 5. Extract ECG

This reads the selected raw recordings, extracts the channel labelled `65`, and
selects the last complete 150–260 s take for each movie. The full ECG and six movie
markers are saved in `<output folder>/ecg/extracted/`. The current HRV method requires 500 Hz data.

In [ ]:
with LOG.step("ECG extraction"):
    if SUBJECTS:
        import numpy as np
        import pandas as pd
        from extract_ecg import extract_one

        (OUTPUTS / "ecg/extracted").mkdir(parents=True, exist_ok=True)
        overwrite = choose_overwrite(
            [OUTPUTS / "ecg/extracted" / f"{sid}_movie_ecg.npz" for sid in SUBJECTS], "ECG extraction"
        )
        rows = []
        for sid in SUBJECTS:
            vhdr = RAW_DATA_DIR / f"{sid}_movie.vhdr"
            output = OUTPUTS / "ecg/extracted" / f"{sid}_movie_ecg.npz"
            inputs = [vhdr.with_suffix(ext) for ext in (".vhdr", ".eeg", ".vmrk")]
            inputs.append(ECG_DIR / "ECG_R_peak_editor/extract_ecg.py")
            if not needs_run(output, inputs, overwrite):
                rows.append({"ID": sid, "status": "Skipped existing", "file": output.name})
                continue
            extract_one(vhdr, output)
            with np.load(output, allow_pickle=False) as ecg:
                if int(ecg["fs"]) != 500:
                    raise ValueError(f"{sid}: this workflow requires 500 Hz recordings.")
                LOG.write(
                    f"{sid} ECG: channel {ecg['ch_name'].item()}, {int(ecg['fs'])} Hz, "
                    f"{int(ecg['n_samples'])} samples, {len(ecg['event_sample'])} movie markers"
                )
                LOG.write(f"Saved: {output}")
                rows.append({"ID": sid, "status": "Created", "samples": int(ecg["n_samples"]),
                             "fs": int(ecg["fs"]), "markers": len(ecg["event_sample"])})
        display(pd.DataFrame(rows))
    else:
        print("No available subjects; skipped.")
        LOG.write("Skipped: no available subjects")


### 6. Reuse or review R peaks

The notebook looks for each selected subject's
`<output folder>/ecg/r_peaks/{ID}_movie_rpeaks.npz` and offers:

- **1 — skip existing files (default):** open the editor only for missing or empty files.
- **2 — review again:** open the editor for the selected subjects, including those
  with saved corrections. Changes replace the old file only when you save.

When needed, the launcher fills in INPUT, OUTPUT and the first ID to review.
Review the full recording and press `q` to save and return to the launcher.
For a batch, enter the next ID printed below. Close the launcher when finished.
[Editor controls](ecg/ECG_R_peak_editor/README.md).

You can also use the [Mac application](ecg/ECG_R_peak_editor/distribution/README.md)
with the INPUT and OUTPUT folders printed below (`<output folder>/ecg/extracted` and `<output folder>/ecg/r_peaks`).
Saving is sufficient; there is no separate ID-confirmation prompt.
Step 7 checks that the saved R peaks match the extracted ECG before computing HRV.


In [ ]:
with LOG.step("R-peak corrections"):
    if not SUBJECTS:
        print("No available subjects; skipped.")
        LOG.write("Skipped: no available subjects")
    else:
        import numpy as np

        peak_files = {sid: OUTPUTS / "ecg/r_peaks" / f"{sid}_movie_rpeaks.npz" for sid in SUBJECTS}
        overwrite = choose_overwrite(
            peak_files.values(), "R-peak corrections",
            replace_label="review again in the editor; replace files when you save",
        )
        review_ids = [sid for sid in SUBJECTS if overwrite or not has_output(peak_files[sid])]
        if not review_ids:
            print("Existing R-peak files kept; editor skipped.")
            LOG.write("Kept existing R-peak files; editor skipped")
        else:
            require_files([OUTPUTS / "ecg/extracted" / f"{sid}_movie_ecg.npz" for sid in review_ids])
            (OUTPUTS / "ecg/r_peaks").mkdir(parents=True, exist_ok=True)
            print("INPUT:", LOG.display(OUTPUTS / "ecg/extracted"))
            print("OUTPUT:", LOG.display(OUTPUTS / "ecg/r_peaks"))
            print("Review these subjects:", ", ".join(review_ids))
            LOG.write("Editor opened for: " + ", ".join(review_ids))
            LOG.command([
                sys.executable, ECG_DIR / "ECG_R_peak_editor/launch_editor.py",
                "--raw-dir", OUTPUTS / "ecg/extracted",
                "--out-dir", OUTPUTS / "ecg/r_peaks",
                "--subject", review_ids[0],
            ], env=dict(os.environ, DBH_EDITOR_STDOUT="1", PYTHONUNBUFFERED="1"))
            missing = [sid for sid in SUBJECTS if not has_output(peak_files[sid])]
            if missing:
                print("Still missing corrected R-peak files:", ", ".join(missing))
                LOG.write("Missing R-peak files: " + ", ".join(missing))
            else:
                print("Saved R-peak files found. Continue to step 7.")
        LOG.write(f"R-peak folder: {OUTPUTS / 'ecg/r_peaks'}")
        for sid, peak_path in peak_files.items():
            if has_output(peak_path):
                with np.load(peak_path, allow_pickle=False) as peaks:
                    LOG.write(f"{sid}: {peaks['r_peaks'].size} saved R peaks")


### 7. Compute IBI and HRV time series

The shared function produces IBI at 4 Hz and LF/HF power at 1 Hz, preserving the
selected CSV order. Single-subject runs save `outputs/demo/ecg/hrv/ecg_ts_{ID}.npz`; batch runs
save `outputs/batch/ecg/hrv/ecg_ts.npz`.

If the output exists, **1** keeps it unchanged; **2** recalculates it for the current
selected subjects and replaces it after successful computation. Choose 2 when
inputs, settings, or the subject list have changed.
Before computing, this step checks the R-peak files against the extracted ECG.
Missing corrections are listed and HRV is skipped until they are saved.


In [ ]:
with LOG.step("IBI and HRV"):
    if not SUBJECTS:
        print("No available subjects; skipped.")
        LOG.write("Skipped: no available subjects")
    else:
        overwrite = choose_overwrite([HRV_OUTPUT], "IBI and HRV time series")
        if has_output(HRV_OUTPUT) and not overwrite:
            print("Existing HRV result kept:", LOG.display(HRV_OUTPUT))
            LOG.write(f"Kept: {HRV_OUTPUT}")
        else:
            from r_peak_files import split_corrected_rpeaks
            import compute_hrv_timeseries as hrv

            _, pending = split_corrected_rpeaks(SUBJECTS, OUTPUTS)
            if pending:
                print("HRV skipped; review and save R peaks for:", ", ".join(pending))
                LOG.write("HRV skipped; missing R-peak corrections: " + ", ".join(pending))
                print("Complete step 6, then rerun this cell.")
            else:
                LOG.write(
                    f"Parameters: ECG {hrv.FS} Hz; drop first {hrv.DROP_SEC} s; "
                    f"analyse {hrv.ANALYSIS_SEC} s; IBI {hrv.RESAMPLE_HZ} Hz; LF/HF 1 Hz; "
                    f"LF {hrv.LF_BAND}, HF {hrv.HF_BAND} Hz; PWVD"
                )
                arrays = hrv.compute_hrv(
                    SUBJECTS, rpeak_dir=OUTPUTS / "ecg/r_peaks",
                    out_npz=HRV_OUTPUT, overwrite=overwrite,
                )
                conditions = "/".join(arrays["conditions"].astype(str))
                for i, sid in enumerate(SUBJECTS):
                    LOG.write(f"{sid} HRV beats ({conditions}): {arrays['n_beats'][i].tolist()}")
                LOG.write(
                    f"Saved: {HRV_OUTPUT}; IBI shape {arrays['ibi'].shape}; "
                    f"LF/HF shape {arrays['lf'].shape}"
                )
                print("HRV output:", LOG.display(HRV_OUTPUT))


## Manuscript analyses

The supplied manuscript preprocessing outputs for all 148 children are in
`01_preprocessing/outputs/manuscript/`: cleaned EEG, corrected R peaks, EEG band
power and HRV. The analysis notebooks read them by default. The single-subject
demonstration reads `outputs/demo/`.

| Analysis | Preprocessing input |
|---|---|
| [Eliciting effects](../02_eliciting_effects/run_eliciting_effects.ipynb) | `outputs/manuscript/eeg/band_power/`, `outputs/manuscript/ecg/hrv/ecg_ts.npz` |
| [Correlation](../03_correlation/run_correlation.ipynb) | `outputs/manuscript/eeg/band_power/`, `outputs/manuscript/ecg/hrv/ecg_ts.npz` |
| [HEP](../04_HEP/run_hep.ipynb) | batch: `outputs/manuscript/eeg/cleaned/`, `outputs/manuscript/ecg/r_peaks/`; demonstration: `outputs/demo/` (cleaned EEG, extracted ECG, R peaks) |
| [dBHI](../05_dBHI/run_dbhi.ipynb) | `outputs/manuscript/eeg/cleaned/`, `outputs/manuscript/ecg/r_peaks/` |

Cohort inputs and setup are described in the [root README](../README.md).
